# 🎭 Face Swap Colab 自动下载版

**适用平台：** Google Colab (免费 T4 GPU)

**本版本特点：** 无需手动上传文件，bundle 通过本地隧道自动下载到 Colab。更适合不想上传大文件到 Colab 的用户。

## ⚠️ 重要前提

1. 本 Mac/Codex 会话必须**保持运行**，直到 Colab 下载完成
2. 需要上传 `cloud_gpu_faceswap_upload.tar.gz`（由 `scripts/00_make_upload_package.sh` 生成）
3. Colab 会话有时间限制，建议准备 2–3 小时完成全流程

## Step 0 — 检查 GPU 环境

⏱ 预计时间：几秒钟

确认 Colab 已分配到 NVIDIA GPU（免费版通常是 T4）。

In [ ]:
!nvidia-smi

In [ ]:
import os
ROOT = '/content/faceswap_work'
os.makedirs(ROOT, exist_ok=True)
print('工作目录:', ROOT)

## Step 1 — 自动下载 Bundle

⏱ 预计时间：1–10 分钟（取决于 bundle 大小和网络速度）

**⚠️ 重要：** 运行此单元格前，必须先在**本地 Mac** 运行以下命令启动本地隧道服务：

```bash
# 在 Mac 本地，进入项目目录
cd cloud_gpu_faceswap

# 安装 npx（如果未安装）
# brew install npx

# 启动本地 HTTP 服务器并暴露给公网
python3 -m http.server 8080 &
npx localtunnel --port 8080
```

运行 `npx localtunnel --port 8080` 后，终端会输出类似 `https://xxxxx.loca.lt/` 的 URL。

将该 URL 中的**子域名**（`xxxxx`）复制到下方代码中替换 `YOUR_LOCALTUNNEL_SUBDOMAIN`。

In [ ]:
# ⬇️ 替换为 localtunnel 的子域名
# 例如：URL 是 https://abc123.loca.lt/，则填入 'abc123'
LOCALTUNNEL_SUBDOMAIN = 'YOUR_LOCALTUNNEL_SUBDOMAIN'

import os, subprocess

BUNDLE_URL = f'https://{LOCALTUNNEL_SUBDOMAIN}.loca.lt/cloud_gpu_faceswap_upload.tar.gz'
ROOT = '/content/faceswap_work'
os.makedirs(ROOT, exist_ok=True)

# 清理旧文件
!rm -rf /content/faceswap_work/cloud_gpu_faceswap /content/faceswap_work/source_faces

# 下载 bundle
print(f'正在从 {BUNDLE_URL} 下载 ...')
!wget -O /content/cloud_gpu_faceswap_upload.tar.gz "$BUNDLE_URL" --timeout=300

# 解压
!tar -xzf /content/cloud_gpu_faceswap_upload.tar.gz -C /content/faceswap_work

# 验证文件
!find /content/faceswap_work -maxdepth 3 -type f | sort | sed -n '1,80p'

## Step 2 — 安装 Faceswap 工具

⏱ 预计时间：10–20 分钟

从 GitHub 克隆 deepfakes/faceswap 仓库，并安装 NVIDIA CUDA 依赖。

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap

apt-get update -y
apt-get install -y ffmpeg git python3-venv

mkdir -p tools
if [ ! -d tools/faceswap/.git ]; then
  echo '正在克隆 faceswap 仓库 ...'
  git clone https://github.com/deepfakes/faceswap.git tools/faceswap
else
  echo 'faceswap 仓库已存在，跳过克隆'
fi

cd tools/faceswap
python3 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip setuptools wheel
python -m pip install -r requirements/requirements_nvidia.txt

python faceswap.py -h >/dev/null
echo '\n✅ faceswap 安装完成！'

## Step 3 — 准备素材并提取人脸

⏱ 预计时间：5–15 分钟

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap
bash scripts/02_prepare_workspace.sh
bash scripts/03_extract_faces.sh

## Step 4 — 人工检查提取的人脸

查看缩略图拼图，确认人脸提取正确。

如果发现错误人脸：
1. 在文件管理器中找到 `/content/faceswap_work/cloud_gpu_faceswap/workspace/`
2. 删除错误的人脸图

✅ 确认无误后，进入下一步训练。

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display

def contact_sheet(folder, title, thumb=128, cols=8):
    """生成缩略图拼图，方便快速检查人脸提取结果"""
    paths = sorted(Path(folder).glob('*'))[:64]
    if not paths:
        print('无图片:', folder)
        return
    rows = (len(paths) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * thumb, rows * (thumb + 22)), 'white')
    draw = ImageDraw.Draw(sheet)
    for idx, path in enumerate(paths):
        img = Image.open(path).convert('RGB')
        img = ImageOps.contain(img, (thumb, thumb))
        x = (idx % cols) * thumb
        y = (idx // cols) * (thumb + 22)
        sheet.paste(img, (x, y))
        draw.text((x + 2, y + thumb + 2), path.name[:18], fill=(0, 0, 0))
    print(title)
    display(sheet)

base = '/content/faceswap_work/cloud_gpu_faceswap/workspace'
contact_sheet(f'{base}/source_faces_extract', '【源人物】人脸提取结果（Source Faces）')
contact_sheet(f'{base}/target_faces_extract', '【目标人物】人脸提取结果（Target Faces）')

## Step 5 — 训练模型

⏱ 预计时间：
- **1000 次迭代**：15–30 分钟（快速测试）
- **3000 次迭代**：45–90 分钟（推荐）
- **5000 次迭代**：90–180 分钟（高质量）

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap
ITERATIONS=3000 bash scripts/04_train_preview.sh

## Step 6 — 转换并下载换脸视频

⏱ 预计时间：3–10 分钟

输出文件：`output/faceswap_test.mp4`

In [ ]:
%%bash
set -euo pipefail
cd /content/faceswap_work/cloud_gpu_faceswap
bash scripts/05_convert_test.sh
ls -lh output/faceswap_test.mp4

In [ ]:
from google.colab import files
files.download('/content/faceswap_work/cloud_gpu_faceswap/output/faceswap_test.mp4')

---

## ❓ 常见问题（FAQ）

### Q1: localtunnel 下载失败
**A：** 
- 确认 Mac 本地的 `python3 -m http.server 8080` 仍在运行
- 确认 `npx localtunnel --port 8080` 终端标签页**保持打开**
- 检查子域名是否正确输入（不含 `.loca.lt`）

### Q2: Colab 访问 localtunnel 时要求输入密码
**A：** 这是正常行为。localtunnel 默认会要求访问者输入其出口 IP。
在浏览器中访问一次 `https://{subdomain}.loca.lt`，按提示输入数字密码后，再重新运行 Colab 单元格。

### Q3: Colab 断连了怎么办？
**A：** 同普通 Colab 版，建议将 `workspace/model` 同步到 Google Drive。

### Q4: 其他问题
**A：** 参考 `faceswap_free_colab.ipynb` 中的 FAQ。